# SWANIRVAR — Backend-Only Final Packager

Run cells 1→5. This packages the **current backend from Google Drive** and all Swanirvar notebooks. It excludes the React frontend, node_modules, and large Drive-wide datasets/models/checkpoints.

In [ ]:
# CELL 1 — Locate current backend
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT=Path('/content/drive/MyDrive/Swanirvar')
PROJECT=DRIVE_ROOT/'Swanirvar_COMPLETE_v1.1.2'
APP=PROJECT/'application'
BACKEND=APP/'backend'
NOTEBOOKS=DRIVE_ROOT/'notebooks'
print('Drive:',DRIVE_ROOT)
print('Backend:',BACKEND)
for p in [BACKEND/'app.py', BACKEND/'core', BACKEND/'services', BACKEND/'data']:
    print(('✅' if p.exists() else '❌'), p)
if not all(p.exists() for p in [BACKEND/'app.py',BACKEND/'core',BACKEND/'services',BACKEND/'data']):
    raise FileNotFoundError('Current Swanirvar backend structure not found.')
print('✅ CELL 1 COMPLETE')


In [ ]:
# CELL 2 — Validate current backend
import compileall, re
ok=compileall.compile_dir(str(BACKEND), quiet=1)
print('✅ compileall passed' if ok else '❌ compileall failed')
if not ok: raise RuntimeError('Backend compilation failed.')
text=(BACKEND/'app.py').read_text(encoding='utf-8',errors='replace')
checks={
 '7D':'/api/v1/7d/' in text,
 'profile':'/api/v1/profile/' in text,
 'ML/DL':'/api/v1/ml-advisory/' in text,
 'market':'/api/v1/market/' in text,
 'evidence':'/api/v1/evidence/' in text,
 'Sathi':'/api/v1/sathi' in text,
 'voice':'/api/v1/voice/' in text,
 'reports':'/api/v1/report' in text,
}
for k,v in checks.items(): print(('✅' if v else '⚠️'),k)
print('Registered route count:', len(re.findall(r'@app\.(?:get|post|put|delete)\(', text)))
print('✅ CELL 2 COMPLETE')


In [ ]:
# CELL 3 — Install deps and smoke-test the current backend
import subprocess,sys,time,socket,urllib.request,urllib.error
pkgs=['fastapi>=0.110,<1','uvicorn[standard]>=0.29,<1','pydantic>=2,<3','python-multipart>=0.0.9,<1','requests>=2.31,<3','reportlab>=4,<5','google-genai>=1,<2']
subprocess.run([sys.executable,'-m','pip','install','-q',*pkgs],check=True)
subprocess.run(['bash','-lc',"fuser -k 8000/tcp 2>/dev/null || true; pkill -f 'uvicorn' 2>/dev/null || true"],check=False)
time.sleep(2)
log=open('/content/swanirvar_backend_final.log','w',encoding='utf-8')
proc=subprocess.Popen([sys.executable,'-m','uvicorn','backend.app:app','--host','127.0.0.1','--port','8000'],cwd=str(APP),stdout=log,stderr=subprocess.STDOUT)
ready=False
for _ in range(60):
    if proc.poll() is not None: break
    try:
        s=socket.create_connection(('127.0.0.1',8000),timeout=1); s.close(); ready=True; break
    except OSError: time.sleep(.4)
if not ready:
    log.flush(); print(open('/content/swanirvar_backend_final.log',encoding='utf-8').read()[-12000:]); raise RuntimeError('Backend failed to start.')
print('✅ FastAPI started')

def get(path):
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000'+path,timeout=20) as r: return r.status,r.read().decode('utf-8',errors='replace')
    except urllib.error.HTTPError as e: return e.code,e.read().decode('utf-8',errors='replace')
for path in ['/health','/api/v1/pilots','/api/v1/profile/WB_RURAL_01','/api/v1/ml-advisory/WB_RURAL_01','/api/v1/7d/WB_RURAL_01','/api/v1/feasibility/WB_RURAL_01']:
    code,body=get(path); print(('✅' if code==200 else '⚠️'),path,'→',code)
    if code!=200: print(body[:1500])
print('✅ CELL 3 COMPLETE')


In [ ]:
# CELL 4 — Create exact current backend-only ZIP
from pathlib import Path
import shutil,zipfile,json
from datetime import datetime
OUT=Path('/content/swanirvar_backend_final_package')
if OUT.exists(): shutil.rmtree(OUT)
OUT.mkdir(parents=True)

def ignore_cache(d,names): return {n for n in names if n in {'__pycache__','.pytest_cache'}}
shutil.copytree(BACKEND, OUT/'backend', ignore=ignore_cache)
(OUT/'requirements.txt').write_text('''fastapi>=0.110,<1
uvicorn[standard]>=0.29,<1
pydantic>=2,<3
python-multipart>=0.0.9,<1
requests>=2.31,<3
reportlab>=4,<5
google-genai>=1,<2
''',encoding='utf-8')
if NOTEBOOKS.exists(): shutil.copytree(NOTEBOOKS, OUT/'notebooks', ignore=ignore_cache)
manifest={'package':'Swanirvar_Backend_ONLY_CURRENT','created_at':datetime.now().isoformat(),'frontend_included':False,'backend_source':str(BACKEND),'notebooks_included':True,'deterministic_7d_preserved':True}
(OUT/'PACKAGE_MANIFEST.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
ZIP=Path('/content/Swanirvar_Backend_ONLY_CURRENT.zip')
if ZIP.exists(): ZIP.unlink()
with zipfile.ZipFile(ZIP,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob('*'):
        if p.is_file(): z.write(p,p.relative_to(OUT).as_posix())
print('✅ Created:',ZIP)
print('Size MB:',round(ZIP.stat().st_size/1024/1024,2))
print('Contains: backend + backend data/web + all notebooks')
print('Excludes: frontend + node_modules + large datasets/models/checkpoints')


In [ ]:
# CELL 5 — Download
from google.colab import files
files.download('/content/Swanirvar_Backend_ONLY_CURRENT.zip')
print('✅ Download requested')
